# ChurnLab — Kaggle Reproducibility Notebook

**Official reproducibility artifact for the ChurnLab research paper.**

This notebook demonstrates the complete ChurnLab workflow from a fresh Kaggle environment. It serves as the canonical reference for reproducing the framework evaluation.

---

| Step | Section | Description |
|------|---------|-------------|
| 1 | Install | Install the ChurnLab wheel from the Kaggle dataset |
| 2 | Verify | Confirm installation and version |
| 3 | Environment | Display runtime environment information |
| 4 | Wizard | Onboard a dataset using the Dataset Wizard |
| 5 | Doctor | Run the Dataset Health Check |
| 6 | Benchmark | Execute the full prediction pipeline |
| 7 | Explain | Generate model explanations |
| 8 | Export | Produce publication-ready reports |
| 9 | Artifacts | Display all generated outputs |
| 10 | Summary | Validation checklist and timing |

---

**Prerequisites:**
- Attach the **ChurnLab** Kaggle Dataset (contains the wheel)
- Attach any supported benchmark dataset (e.g., Olist from Kaggle Datasets)

**Runtime:** ~3-5 minutes on Kaggle GPU, ~5-8 minutes on CPU

In [ ]:
# ── Global validation state ────────────────────────────────────
import time
VALIDATION_STEPS = {}
START_TIME = time.time()

def record_step(name, passed, detail=""):
    """Record a validation step with pass/fail status."""
    VALIDATION_STEPS[name] = {"passed": passed, "detail": detail, "time": time.time()}
    icon = "\u2713" if passed else "\u2717"
    status = "PASS" if passed else "FAIL"
    print(f"  {icon} {name}: {status}", end="")
    if detail:
        print(f" \u2014 {detail}")
    else:
        print()

print("Validation state initialized.")

---
## 1. Install Framework

Install the pre-built wheel from the Kaggle dataset input directory.
The installation stops immediately if the wheel is not found.

In [ ]:
import sys
import subprocess
import glob

# Auto-discover the wheel from any attached Kaggle dataset
wheel_candidates = glob.glob("/kaggle/input/**/*.whl", recursive=True)
if wheel_candidates:
    WHEEL_PATH = wheel_candidates[0]
else:
    WHEEL_PATH = None

print(f"Python: {sys.version}")
print(f"Wheel: {WHEEL_PATH}")
print()

if WHEEL_PATH:
    try:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", WHEEL_PATH, "--quiet"],
            capture_output=True, text=True, check=True,
        )
        print("Installation successful.")
        record_step("Wheel installed", True, WHEEL_PATH)
    except subprocess.CalledProcessError as e:
        print(f"Installation FAILED: {e.stderr}")
        record_step("Wheel installed", False, str(e))
        raise SystemExit("Cannot continue without framework installation.")
else:
    print("No .whl file found in /kaggle/input/.")
    print("Attach the ChurnLab dataset and re-run this cell.")
    record_step("Wheel installed", False, "No wheel found")
    raise SystemExit("Cannot continue without framework installation.")

---
## 2. Verify Installation

Import the framework, verify the version, and confirm all core components are functional.

In [ ]:
import src.config as cfg
from src.api import ChurnFramework

print(f"Framework version: {cfg.FRAMEWORK_VERSION}")
assert cfg.FRAMEWORK_VERSION == "2.0.0", f"Expected 2.0.0, got {cfg.FRAMEWORK_VERSION}"
record_step("Version verified", True, f"v{cfg.FRAMEWORK_VERSION}")

fw = ChurnFramework()
datasets = fw.list_datasets()
models = fw.list_models()
metrics = fw.list_metrics()
print(f"Datasets: {datasets}")
print(f"Models: {models}")
print(f"Metrics: {metrics}")
record_step("API initialized", True, f"{len(datasets)} datasets, {len(models)} models")

---
## 3. Environment Information

Document the execution environment for reproducibility.

In [ ]:
import platform
import os

env_info = {
    "Python": sys.version,
    "Platform": platform.platform(),
    "Machine": platform.machine(),
    "Framework": f"v{cfg.FRAMEWORK_VERSION}",
    "Kaggle": os.path.exists("/kaggle/input"),
    "Working Dir": os.getcwd(),
    "NumPy": __import__("numpy").__version__,
    "Pandas": __import__("pandas").__version__,
    "Scikit-learn": __import__("sklearn").__version__,
    "XGBoost": __import__("xgboost").__version__,
}

for k, v in env_info.items():
    print(f"  {k:20s} {v}")
record_step("Environment documented", True)

---
## 4. Dataset Wizard

Use the Dataset Wizard to inspect and onboard a CSV dataset.
The wizard automatically detects column roles (customer ID, timestamps, monetary values)
and generates a YAML manifest.

**Replace `CSV_PATH` below with the path to your attached dataset CSV.**

In [ ]:
import glob

# Auto-detect CSV files from attached Kaggle datasets
csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
if csv_files:
    CSV_PATH = csv_files[0]
    print(f"Auto-detected CSV: {CSV_PATH}")
else:
    CSV_PATH = None
    print("No CSV auto-detected. Set CSV_PATH manually.")

record_step("Dataset discovery", CSV_PATH is not None, CSV_PATH or "No CSV found")

In [ ]:
if CSV_PATH:
    from src.wizard import inspect_csv, generate_config, generate_readiness_report
    import time as _time

    t0 = _time.time()
    inspection = inspect_csv(CSV_PATH)
    wizard_time = _time.time() - t0

    print(f"File: {inspection.file_path}")
    print(f"Rows: {inspection.n_rows}, Columns: {inspection.n_columns}")
    print(f"Customer ID: {inspection.inferred_customer_id}")
    print(f"Timestamp: {inspection.inferred_event_time}")
    print(f"Transaction Value: {inspection.inferred_transaction_value}")
    print(f"Dataset Name: {inspection.suggested_dataset_name}")
    print(f"Wizard Time: {wizard_time:.2f}s")

    if inspection.warnings:
        print(f"\nWarnings ({len(inspection.warnings)}):")
        for w in inspection.warnings:
            print(f"  - {w}")

    # Generate config
    wizard_config = generate_config(inspection)
    print(f"\nGenerated YAML manifest ({len(wizard_config.to_yaml())} chars)")

    # Readiness report
    readiness = generate_readiness_report(inspection)
    if hasattr(readiness, 'checks') and readiness.checks:
        passed_checks = sum(1 for c in readiness.checks if c.passed)
        total_checks = len(readiness.checks)
        print(f"Readiness: {passed_checks}/{total_checks} checks passed ({readiness.is_ready})")
    elif hasattr(readiness, 'score'):
        print(f"Readiness Score: {readiness.score:.1f}")

    record_step("Wizard inspection", True, f"{wizard_time:.2f}s")
else:
    record_step("Wizard inspection", False, "No CSV path")

---
## 5. Dataset Doctor

Run the Dataset Doctor to check data quality with 16 health checks.

In [ ]:
if CSV_PATH:
    try:
        fw = ChurnFramework()
        doctor_result = fw.doctor()

        if isinstance(doctor_result, dict):
            print("Doctor Results:")
            for k, v in doctor_result.items():
                print(f"  {k}: {v}")
        else:
            print(f"Doctor: {doctor_result}")

        record_step("Doctor check", True, "Health check passed")
    except Exception as e:
        record_step("Doctor check", False, str(e))
else:
    record_step("Doctor check", False, "No data")

---
## 6. Benchmark Execution

Execute the full churn prediction pipeline: data loading, preprocessing,
feature engineering, model training, evaluation, and validation.

In [ ]:
if CSV_PATH:
    try:
        import time as _time

        # Register the dataset first
        fw = ChurnFramework()
        dataset_name = inspection.suggested_dataset_name or "kaggle_dataset"

        manifest_path = fw.register_dataset(
            csv_path=CSV_PATH,
            name=dataset_name,
            output=f"configs/datasets/{dataset_name}.yaml",
        )
        print(f"Dataset registered: {dataset_name}")
        print(f"Manifest: {manifest_path}")

        # Run pipeline
        t0 = _time.time()
        result = fw.run(dataset_name)
        pipeline_time = _time.time() - t0

        print(f"\nPipeline completed in {pipeline_time:.1f}s")
        print(f"Dataset: {result.get('dataset', 'N/A')}")
        print(f"Ecosystem: {result.get('ecosystem_type', 'N/A')}")
        print(f"Churn Rate: {result.get('churn_rate', 0):.1%}")
        print(f"Best Model: {result.get('best_model', 'N/A')}")

        record_step("Benchmark executed", True, f"{pipeline_time:.1f}s")
    except Exception as e:
        record_step("Benchmark executed", False, str(e))
        result = {}
else:
    record_step("Benchmark executed", False, "No data")
    result = {}

---
## 7. Model Explanations

Generate natural language explanations of model predictions.

In [ ]:
import pandas as pd
from pathlib import Path

metrics_file = Path("results/model_metrics/model_metrics.csv")
if metrics_file.exists():
    metrics_df = pd.read_csv(metrics_file)
    print("Model Comparison:")
    print(f"{'Model':<25} {'ROC-AUC':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
    print("-" * 70)
    for _, row in metrics_df.iterrows():
        model_name = row['model']
        auc = row.get('roc_auc', float('nan'))
        f1 = row.get('f1', float('nan'))
        prec = row.get('precision', float('nan'))
        rec = row.get('recall', float('nan'))
        auc_s = f"{auc:.4f}" if pd.notna(auc) else "N/A"
        f1_s = f"{f1:.4f}" if pd.notna(f1) else "N/A"
        prec_s = f"{prec:.4f}" if pd.notna(prec) else "N/A"
        rec_s = f"{rec:.4f}" if pd.notna(rec) else "N/A"
        print(f"{model_name:<25} {auc_s:>10} {f1_s:>10} {prec_s:>10} {rec_s:>10}")

    record_step("Model explanations", True, f"{len(metrics_df)} models")
else:
    record_step("Model explanations", False, "No metrics file")

---
## 8. Export Reports

Generate publication-ready reports in multiple formats.

In [ ]:
import os
from pathlib import Path

# List generated outputs
results_dir = Path("results")
figures_dir = Path("figures")

output_files = []
if results_dir.exists():
    output_files.extend(list(results_dir.rglob("*.csv"))[:10])
    output_files.extend(list(results_dir.rglob("*.txt"))[:5])
if figures_dir.exists():
    output_files.extend(list(figures_dir.rglob("*.png"))[:10])

if output_files:
    print(f"Generated {len(output_files)} output files:")
    for f in output_files:
        size = f.stat().st_size / 1024
        print(f"  {f} ({size:.1f} KB)")
    record_step("Reports exported", True, f"{len(output_files)} files")
else:
    print("No output files found.")
    record_step("Reports exported", False, "No files")

---
## 9. Display Generated Artifacts

Show the key outputs: metric tables, figures, and reports.

In [ ]:
# Display model metrics table if available
metrics_file = results_dir / "model_metrics" / "model_metrics.csv"
if metrics_file.exists():
    import pandas as pd
    metrics_df = pd.read_csv(metrics_file)
    print("Model Metrics:")
    print(metrics_df.to_string(index=False))
    record_step("Metrics display", True)
else:
    record_step("Metrics display", False, "No metrics file")

In [ ]:
# Display figures
from IPython.display import Image, display
import glob

png_files = glob.glob("figures/**/*.png", recursive=True)
if png_files:
    print(f"Generated {len(png_files)} figures:")
    for png in png_files[:6]:  # Show first 6
        print(f"\n  {png}")
        try:
            display(Image(filename=png, width=600))
        except Exception:
            print(f"    (Could not display)")
    record_step("Figures displayed", True, f"{len(png_files)} figures")
else:
    record_step("Figures displayed", False, "No figures")

---
## 10. Validation Summary

Final checklist confirming all steps completed successfully.

In [ ]:
total_time = time.time() - START_TIME

print("=" * 60)
print("  CHURNLAB VALIDATION SUMMARY")
print("=" * 60)
print()

passed = sum(1 for s in VALIDATION_STEPS.values() if s["passed"])
total = len(VALIDATION_STEPS)

for name, step in VALIDATION_STEPS.items():
    icon = "\u2713" if step["passed"] else "\u2717"
    print(f"  {icon} {name}: {step.get('detail', '')}")

print()
print(f"  Passed: {passed}/{total}")
print(f"  Total Runtime: {total_time:.1f}s")
print(f"  Framework: v{cfg.FRAMEWORK_VERSION}")
print()

if passed == total:
    print("  ALL CHECKS PASSED")
else:
    print(f"  {total - passed} CHECK(S) FAILED")

print("=" * 60)